# Direct Preference Optimization — a small CPU example

Python 3.13: `python -m pip install -r requirements.txt`. Run all cells in order. This uses the official pinned SmolLM2-135M-Instruct checkpoint and LoRA; no hosted API, GPU or token is required. Eight author-created arithmetic pairs teach the DPO mechanics. Two disjoint prompts provide a small held-out loss check, not a model-quality benchmark or reproduction of the DPO paper. No PPO implementation is implied. [Read Part 1](https://todatabeyond.com/blog/visual-guide-to-llm-preference-tuning-with-rlhf-and-ppo).

In [ ]:
import copy,hashlib,tempfile
from pathlib import Path
import torch
from torch.nn import functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM
from peft import LoraConfig,get_peft_model,PeftModel
torch.set_num_threads(4)
torch.manual_seed(7)
MODEL='HuggingFaceTB/SmolLM2-135M-Instruct'
REVISION='12fd25f77366fa6b3b4b768ec3050bf629380bac'
print('CPU; pinned pretrained model:',MODEL)

In [ ]:
# Author-created miniature preference pairs, not a benchmark or human study.
train=[('2 + 2','4','5'),('3 + 1','4','7'),('5 - 2','3','8'),('2 * 3','6','9'),
       ('6 - 1','5','2'),('4 + 3','7','6'),('8 - 2','6','5'),('3 * 3','9','8')]
heldout=[('7 + 2','9','4'),('9 - 2','7','3')]
assert {p for p,_,_ in train}.isdisjoint(p for p,_,_ in heldout)
assert all(chosen!=rejected for _,chosen,rejected in train+heldout)
tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REVISION,token=False)
tokenizer.pad_token=tokenizer.eos_token
def batch(pairs):
    sequences=[];starts=[]
    for expression,chosen,rejected in pairs:
        prompt='Reply with only the number: '+expression+'.'
        prefix=tokenizer.apply_chat_template([{'role':'user','content':prompt}],add_generation_prompt=True,tokenize=True,return_dict=False)
        for response in (chosen,rejected):
            completion=tokenizer.encode(response+tokenizer.eos_token,add_special_tokens=False)
            sequences.append(prefix+completion);starts.append(len(prefix))
    size=max(map(len,sequences));ids=torch.full((len(sequences),size),tokenizer.pad_token_id,dtype=torch.long)
    attention=torch.zeros_like(ids);mask=torch.zeros_like(ids,dtype=torch.bool)
    for i,(sequence,start) in enumerate(zip(sequences,starts)):
        ids[i,:len(sequence)]=torch.tensor(sequence);attention[i,:len(sequence)]=1
        mask[i,start:len(sequence)]=True
    assert not mask[:,0].any() and mask.any(dim=1).all()
    return ids,attention,mask
train_batch=batch(train);heldout_batch=batch(heldout)
print('Training pairs:',len(train),'; held-out pairs:',len(heldout))

In [ ]:
base=AutoModelForCausalLM.from_pretrained(MODEL,revision=REVISION,token=False,dtype=torch.float32,attn_implementation='eager').to('cpu')
base.config.use_cache=False
model=get_peft_model(base,LoraConfig(r=4,lora_alpha=8,lora_dropout=0,target_modules=['q_proj','v_proj'],task_type='CAUSAL_LM'))
def frozen_hash():
    digest=hashlib.sha256()
    for name,p in model.named_parameters():
        if not p.requires_grad:digest.update(name.encode());digest.update(p.detach().contiguous().numpy().tobytes())
    return digest.hexdigest()
before=frozen_hash()
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable adapter parameters:',trainable)

In [ ]:
def sequence_logps(logits,ids,completion_mask):
    if logits.shape[:2]!=ids.shape or completion_mask.shape!=ids.shape:
        raise ValueError('Mismatched token shapes')
    next_logps=logits[:,:-1].log_softmax(-1).gather(-1,ids[:,1:,None]).squeeze(-1)
    return (next_logps*completion_mask[:,1:]).sum(-1)
def scores(data):
    ids,attention,mask=data
    output=model(input_ids=ids,attention_mask=attention,use_cache=False)
    return sequence_logps(output.logits,ids,mask)
model.eval()
with torch.no_grad(),model.disable_adapter():
    reference_train=scores(train_batch).detach()
    reference_heldout=scores(heldout_batch).detach()
    ids,attention,mask=train_batch
    logits=model(input_ids=ids,attention_mask=attention).logits
    # Independent per-token cross-entropy excludes prompt and padding.
    token_nll=F.cross_entropy(logits[:,:-1].reshape(-1,logits.shape[-1]),ids[:,1:].reshape(-1),reduction='none').reshape(ids.shape[0],-1)
    independent=-(token_nll*mask[:,1:]).sum(-1)
    torch.testing.assert_close(independent,reference_train)
print('Completion-only sequence scores match independent cross-entropy.')

In [ ]:
def dpo_loss(policy,reference,beta=0.1):
    if policy.ndim!=1 or policy.shape!=reference.shape or policy.numel()%2 or policy.numel()==0:
        raise ValueError('Expected nonempty interleaved chosen/rejected pairs')
    if not (0<float(beta)<float('inf')) or not torch.isfinite(policy).all() or not torch.isfinite(reference).all():
        raise ValueError('Finite scores and positive finite beta required')
    if reference.requires_grad:raise ValueError('Reference scores must be frozen')
    margin=(policy[0::2]-policy[1::2])-(reference[0::2]-reference[1::2])
    return -F.logsigmoid(beta*margin).mean(),margin
initial_loss,initial_margin=dpo_loss(reference_train,reference_train)
assert abs(initial_loss.item()-__import__('math').log(2))<1e-6
torch.testing.assert_close(-F.logsigmoid(0.1*initial_margin),F.softplus(-0.1*initial_margin))
print('Initial DPO loss:',round(initial_loss.item(),6))

In [ ]:
optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=5e-4)
losses=[]
for step in range(12):
    model.train();optimizer.zero_grad(set_to_none=True)
    loss,margin=dpo_loss(scores(train_batch),reference_train)
    loss.backward()
    norm=torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
    assert torch.isfinite(norm) and norm>0
    optimizer.step();losses.append(float(loss.detach()))
model.eval()
with torch.no_grad():
    final_loss,final_margin=dpo_loss(scores(train_batch),reference_train)
    validation_loss,validation_margin=dpo_loss(scores(heldout_batch),reference_heldout)
assert final_loss<initial_loss and final_margin.mean()>initial_margin.mean()
assert frozen_hash()==before
print('Training loss:',round(initial_loss.item(),6),'->',round(final_loss.item(),6))
print('Held-out loss:',round(validation_loss.item(),6))
print('Mean training relative log-ratio margin:',round(final_margin.mean().item(),6))
print('Frozen base weights unchanged:',frozen_hash()==before)

In [ ]:
for beta in [0,-1,float('nan'),float('inf')]:
    try:dpo_loss(reference_train,reference_train,beta)
    except ValueError:pass
    else:raise AssertionError('Invalid beta accepted')
for values in [torch.zeros(3),torch.tensor([float('nan'),0.])]:
    try:dpo_loss(values,values)
    except ValueError:pass
    else:raise AssertionError('Invalid pair scores accepted')
with torch.no_grad():
    ids,attention,mask=train_batch
    logits=model(input_ids=ids,attention_mask=attention).logits
    original=sequence_logps(logits,ids,mask)
    modified=logits.clone()
    # Logit positions predict the *next* token. Alter excluded positions only.
    excluded=~mask[:,1:]
    modified[:,:-1][excluded]=torch.randn_like(modified[:,:-1][excluded])*5
    torch.testing.assert_close(sequence_logps(modified,ids,mask),original)
    sample_policy=torch.tensor([-1.,-3.]);sample_ref=torch.tensor([-2.,-2.])
    loss,margin=dpo_loss(sample_policy,sample_ref)
    torch.testing.assert_close(loss,F.softplus(-.1*margin).mean())
print('Prompt/padding exclusion, beta, shape, finite-score and loss-equivalence checks passed.')

In [ ]:
with tempfile.TemporaryDirectory(prefix='dpo-adapter-') as tmp:
    model.save_pretrained(tmp)
    fresh_base=AutoModelForCausalLM.from_pretrained(MODEL,revision=REVISION,token=False,dtype=torch.float32,attn_implementation='eager').to('cpu')
    restored=PeftModel.from_pretrained(fresh_base,tmp).eval()
    ids,attention,mask=train_batch
    with torch.no_grad():
        actual=sequence_logps(restored(input_ids=ids,attention_mask=attention).logits,ids,mask)
        expected=scores(train_batch)
    torch.testing.assert_close(actual,expected,atol=1e-5,rtol=1e-5)
    print('Adapter save/reload maximum score difference:',float((actual-expected).abs().max()))
print('Temporary adapter removed; no model upload performed.')

## Original archive appendix

Original material below is preserved for comparison, not executed.

The complete original draft, figures and unresolved original citation markers are preserved in [Original_DPO_Draft.md](./Original_DPO_Draft.md).